In [1]:
import os
import asyncio
import asyncpg
from dotenv import load_dotenv
from urllib.parse import urlparse, parse_qs, urlencode, urlunparse

load_dotenv()
DATABASE_URL = os.getenv('DATABASE_URL')
if (DATABASE_URL is None):
    raise ValueError("DATABASE_URL is not set in environment variables")


def remove_schema_param(url):
    """Remove the schema parameter from DATABASE_URL for asyncpg"""
    parsed = urlparse(url)
    query_params = parse_qs(parsed.query)

    # Remove 'schema' parameter if it exists
    query_params.pop('schema', None)

    # Rebuild the URL without schema parameter
    new_query = urlencode(query_params, doseq=True)
    new_parsed = parsed._replace(query=new_query)
    return urlunparse(new_parsed)

asyncpg_url = remove_schema_param(DATABASE_URL)
conn = await asyncpg.connect(dsn=asyncpg_url)

In [2]:
import pandas as pd
# fetch all rows and convert to DataFrame
event_rows = await conn.fetch('SELECT id, title, description FROM "Event";')
events_df = pd.DataFrame([dict(r) for r in event_rows])
print(f'Loaded {len(events_df)} rows into comment_df')
events_df.set_index('id', inplace=True)
events_df.head()

Loaded 65056 rows into comment_df


,title,description
id,,
27802,"Solana Up or Down - June 21, 7AM ET","This market will resolve to ""Up"" if the close ..."
27803,"Solana Up or Down - June 21, 6PM ET","This market will resolve to ""Up"" if the close ..."
27804,"Solana Up or Down - June 21, 10PM ET","This market will resolve to ""Up"" if the close ..."
27805,"Solana Up or Down - June 21, 12AM ET","This market will resolve to ""Up"" if the close ..."
27806,"Solana Up or Down - June 21, 3AM ET","This market will resolve to ""Up"" if the close ..."


In [3]:
from bertopic import BERTopic
topic_model = BERTopic(embedding_model="all-MiniLM-L6-v2")

/Users/dhanna/miniconda3/envs/polymarket/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import pandas as pd
# fetch all rows and convert to DataFrame
user_event_associations = await conn.fetch('SELECT * FROM user_event_associations;')
user_event_associations_df = pd.DataFrame([dict(r) for r in user_event_associations])
print(f'Loaded {len(user_event_associations_df)} rows into user_event_associations_df_df')
user_event_associations_df.head()

Loaded 483767 rows into user_event_associations_df_df


,address,name,pseudonym,event_id,event_title,event_slug,event_description
0,0xf3dcaeb909d3d17318c0908a1ec0ca31321e1672,LifelsBeautiful,Weak-Dinner,21617,NL MVP,mlb-nl-mvp,This is a market on predicting the winner of t...
1,0xf3dcaeb909d3d17318c0908a1ec0ca31321e1672,LifelsBeautiful,Weak-Dinner,21618,AL Cy Young,mlb-al-cy-young,This is a market on predicting the winner of t...
2,0xf3dcaeb909d3d17318c0908a1ec0ca31321e1672,LifelsBeautiful,Weak-Dinner,21628,NL Cy Young,mlb-nl-cy-young,This is a market on predicting the winner of t...
3,0xf3dcaeb909d3d17318c0908a1ec0ca31321e1672,LifelsBeautiful,Weak-Dinner,21671,MLB Home Run Leader,mlb-home-run-leader,This is a market on predicting the player with...
4,0xf3dcaeb909d3d17318c0908a1ec0ca31321e1672,LifelsBeautiful,Weak-Dinner,21734,Next Government of Canada,next-government-of-canada,The 2025 Canadian federal election will be hel...


In [5]:
# Create events_df from user_event_associations_df
# Extract unique events with their titles and descriptions
events_from_users = user_event_associations_df[['event_id', 'event_title', 'event_description']].drop_duplicates()

# Rename columns to match original events_df structure
events_from_users = events_from_users.rename(columns={
    'event_id': 'id',
    'event_title': 'title',
    'event_description': 'description'
})

# Convert id to int and set as index
events_from_users['id'] = events_from_users['id'].astype(int)
events_from_users = events_from_users.set_index('id')

print(f'Extracted {len(events_from_users)} unique events from user associations')
print(f'Event ID range: {events_from_users.index.min()} to {events_from_users.index.max()}')
print()
events_from_users.head()

Extracted 33394 unique events from user associations
Event ID range: 3854 to 903799



,title,description
id,,
21617,NL MVP,This is a market on predicting the winner of t...
21618,AL Cy Young,This is a market on predicting the winner of t...
21628,NL Cy Young,This is a market on predicting the winner of t...
21671,MLB Home Run Leader,This is a market on predicting the player with...
21734,Next Government of Canada,The 2025 Canadian federal election will be hel...


In [7]:
from sentence_transformers import SentenceTransformer

# Load the same embedding model used by topic_model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")


print(f"Original events_df: {len(events_df)} events")
print(f"Events from user associations: {len(events_from_users)} events")

# Concatenate the dataframes
# Use concat to combine them, keeping only events that don't already exist in events_df
combined_events = pd.concat([events_df, events_from_users])

# Remove duplicates (keep first occurrence - from original events_df)
combined_events = combined_events[~combined_events.index.duplicated(keep='first')]


# Update events_df to the combined version
events_df = combined_events

# Create combined text for embeddings
events_df['combined_text'] = events_df['title'].fillna('') + " " + events_df['description'].fillna('')

embeddings = embedding_model.encode(events_df['combined_text'].tolist(), show_progress_bar=True)

print(f"\nEmbeddings shape: {embeddings.shape}")
print(f"Number of events with embeddings: {len(embeddings)}")

Original events_df: 65056 events
Events from user associations: 33394 events


Batches: 100%|██████████| 3077/3077 [03:38<00:00, 14.11it/s]



Embeddings shape: (98450, 384)
Number of events with embeddings: 98450


In [8]:
import faiss
import numpy as np

embedding_dim = embeddings.shape[1]
index = faiss.IndexFlatIP(embedding_dim)

normalized_embeddings = embeddings / np.linalg.norm(embeddings, axis=1, keepdims=True)
index.add(normalized_embeddings.astype('float32'))

index_to_id = {i: id for i, id in enumerate(events_df.index)}
id_to_index = {id: i for i, id in enumerate(events_df.index)}

def find_similar_events(event_id, k=5):
    idx = id_to_index[event_id]
    query = normalized_embeddings[idx:idx+1].astype('float32')

    # Search for k+1 because the event itself will be included
    distances, indices = index.search(query, k + 1)

    results = []
    for dist, idx in zip(distances[0], indices[0]):
        similar_id = index_to_id[idx]
        if similar_id != event_id:  # Exclude the query event itself
            results.append({
                'id': similar_id,
                'title': events_df.loc[similar_id, 'title'],
                'similarity': float(dist)
            })
    return results[:k]

In [9]:
k = 5
example_id = events_df.index[6000]
print(f"Finding events similar to: {events_df.loc[example_id, 'title']}\n")

similar_events = find_similar_events(example_id, k=k)
for i, event in enumerate(similar_events, 1):
    print(f"{i}. {event['title']} (similarity: {event['similarity']:.4f})")

Finding events similar to: Charlton Athletic FC vs. West Bromwich Albion FC

1. Charlton Athletic FC vs. West Bromwich Albion FC (similarity: 1.0000)
2. Charlton Athletic FC vs. Southampton FC (similarity: 0.9306)
3. Ipswich Town FC vs. Charlton Athletic FC (similarity: 0.9061)
4. Charlton Athletic FC vs. Swansea City AFC (similarity: 0.9051)
5. Charlton Athletic FC vs. Swansea City AFC (similarity: 0.9051)


In [16]:
# Get a specific user (using the first user as an example)
example_user = user_event_associations_df.iloc[0]['address']
example_name = user_event_associations_df.iloc[0]['name']

print(f"User: {example_name} ({example_user})")
print()

# Get all event_ids for this user
user_events = user_event_associations_df[user_event_associations_df['address'] == example_user]
event_ids = user_events['event_id'].unique()

print(f"Total events betted on: {len(event_ids)}")
print(f"\nEvent IDs: {event_ids}")
print(f"\nSample of events:")
print(user_events[['event_id', 'event_title']].head(10))


def recommend_events_for_user(user_address, top_n=10, similar_per_event=5):
    # Get all events the user has betted on
    user_events = user_event_associations_df[user_event_associations_df['address'] == user_address]
    user_event_ids = set(user_events['event_id'].astype(str).unique())

    # Dictionary to accumulate scores for candidate events
    event_scores = {}

    # For each event the user has betted on, find similar events
    for event_id in user_event_ids:
        event_id_int = int(event_id)
        if event_id_int not in events_df.index:
            continue

        # Find similar events
        similar_events = find_similar_events(event_id_int, k=similar_per_event)

        # Add to scores (weight by similarity)
        for similar_event in similar_events:
            similar_id = str(similar_event['id'])
            similarity = similar_event['similarity']

            if similar_id in user_event_ids:
                continue

            if similar_id not in event_scores:
                event_scores[similar_id] = {
                    'id': similar_event['id'],
                    'title': similar_event['title'],
                    'score': 0,
                    'count': 0
                }
            event_scores[similar_id]['score'] += similarity
            event_scores[similar_id]['count'] += 1

    recommendations = sorted(event_scores.values(), key=lambda x: x['score'], reverse=True)[:top_n]

    # Format results
    for rec in recommendations:
        rec['avg_similarity'] = rec['score'] / rec['count']
        rec['total_score'] = rec['score']

    return recommendations

print(f"\nRecommended events for this user::")
recommendations= recommend_events_for_user(example_user, top_n=10, similar_per_event=5)
if recommendations:
    for rank, rec in enumerate(recommendations, 1):
        print(f"{rank}. [{rec['id']}] {rec['title']}")


User: LifelsBeautiful (0xf3dcaeb909d3d17318c0908a1ec0ca31321e1672)

Total events betted on: 226

Event IDs: ['21617' '21618' '21628' '21671' '21734' '21902' '22448' '22976' '22987'
 '23246' '23639' '23834' '24087' '24382' '24404' '24893' '25044' '25315'
 '25323' '25600' '25618' '25929' '25930' '26089' '27174' '27199' '27254'
 '27592' '27824' '28214' '28223' '28261' '28399' '28640' '28812' '28997'
 '28999' '29002' '29007' '29011' '29015' '29036' '29044' '29402' '30102'
 '30103' '30110' '30353' '30386' '30631' '30648' '30806' '31099' '31406'
 '31414' '31418' '31419' '31420' '31426' '31427' '31428' '31433' '31552'
 '31728' '31730' '31736' '31747' '31908' '31909' '32238' '32239' '32322'
 '32434' '32563' '32779' '33495' '33842' '33989' '34050' '35090' '35442'
 '35754' '35774' '35775' '35950' '35951' '36122' '37579' '38101' '38712'
 '40991' '41136' '42575' '42599' '43827' '43877' '44345' '44692' '46203'
 '46724' '48110' '50332' '50925' '51489' '51493' '52169' '52521' '52528'
 '52529' '52532'

In [ ]:
# Generate recommendations for ALL users
from tqdm.auto import tqdm

# Get all unique users
unique_users = user_event_associations_df[['address', 'name', 'pseudonym']].drop_duplicates()
print(f"Total unique users: {len(unique_users)}")
print()

# Store all recommendations
all_recommendations = []

# Process each user
for idx, user_row in tqdm(unique_users.iterrows(), total=len(unique_users), desc="Generating recommendations"):
    user_address = user_row['address']
    user_name = user_row['name']
    user_pseudonym = user_row['pseudonym']
    
    # Generate recommendations for this user
    recommendations = recommend_events_for_user(user_address, top_n=10, similar_per_event=5)
    
    # Store results for each recommended event
    for rank, rec in enumerate(recommendations, 1):
        all_recommendations.append({
            'user_address': user_address,
            'user_name': user_name,
            'user_pseudonym': user_pseudonym,
            'rank': rank,
            'recommended_event_id': rec['id'],
            'recommended_event_title': rec['title'],
            'total_score': rec['total_score'],  # Higher = similar to more user events
            'avg_similarity': rec['avg_similarity'],
            'mention_count': rec['count']
        })

# Convert to DataFrame
recommendations_df = pd.DataFrame(all_recommendations)
print(f"\nGenerated {len(recommendations_df)} total recommendations")
print(f"Average recommendations per user: {len(recommendations_df) / len(unique_users):.2f}")
print()
print("Sample recommendations:")
recommendations_df.head(20)

Total unique users: 24640



Generating recommendations:   1%|          | 214/24640 [00:18<26:03, 15.62it/s]  